# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [6]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [7]:
from langchain_community.document_loaders import PyPDFLoader
import requests


pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
pdf_path = "The GENAI Divide: State of AI in Business 2025.pdf"

response = requests.get(pdf_url, timeout=30)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages.")
print(document_text[:1000])

Loaded 26 pages.
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confi

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

class ArticleSummary(BaseModel):
    Author: str = Field(description="The author of the article.")
    Title: str = Field(description="The title of the article.")
    Relevance: str = Field(description="A one-paragraph explanation of why this article is relevant for an AI professional.")
    Summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens.")
    Tone: str = Field(description="The tone used to produce the summary.")
    InputTokens: int = Field(default=0, description="Number of input tokens used.")
    OutputTokens: int = Field(default=0, description="Number of output tokens generated.")

model_name = "gpt-4o-mini"
selected_tone = "Formal Academic Writing"

developer_instructions = """
You are an expert academic summarizer. Your task is to summarize professional and managerial articles
in a clear, accurate, and concise manner. You must preserve the author's main argument, avoid unsupported
claims, and write in the requested tone.
"""

user_prompt_template = """
Please analyze the following article and produce a structured output.

Requirements:
1. Identify the author.
2. Identify the title.
3. Explain why the article is relevant for an AI professional's professional development.
4. Write a concise and succinct summary no longer than 1000 tokens.
5. Use the following tone: {tone}.

Article:
{article_text}
"""

user_prompt = user_prompt_template.format(
    tone=selected_tone,
    article_text=document_text
)

response = client.beta.chat.completions.parse(
    model=model_name,
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary,
)

summary_obj = response.choices[0].message.parsed

summary_obj.InputTokens = response.usage.prompt_tokens
summary_obj.OutputTokens = response.usage.completion_tokens

summary_text = summary_obj.Summary

summary_obj

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

I evaluate the summary along four dimensions: summarization quality, coherence, tonality, and safety. These criteria are appropriate because The GenAI Divide is an evidence-based business report about AI implementation, organizational adoption, and the gap between pilot projects and measurable business impact. The evaluation questions are designed to check whether the summary captures the report's main findings, explains the business relevance for AI professionals, remains logically organized, follows the requested formal academic tone, and avoids unsupported or misleading claims.

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
import pandas as pd
import os

# Use only the beginning of the article for evaluation to avoid extremely slow DeepEval calls.
# This is enough for the assignment and prevents the evaluator from repeatedly processing the full PDF.
evaluation_source_text = document_text[:12000]

evaluation_model = GPTModel(
    model=os.getenv("MODEL", "gpt-4o-mini"),
    temperature=0,
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

summarization_questions = [
    "Does the summary accurately identify the main argument of The GenAI Divide?",
    "Does the summary explain the gap between AI pilot projects and measurable business impact?",
    "Does the summary discuss the report's evidence base, including surveys, interviews, or disclosed AI initiatives?",
    "Does the summary capture the organizational and managerial implications of GenAI implementation?",
    "Does the summary avoid adding claims that are not supported by the original report?"
]

coherence_questions = [
    "Is the summary logically organized from the report's main problem to its implications?",
    "Are the key ideas connected clearly and smoothly?",
    "Does the summary avoid unnecessary repetition?",
    "Does the summary distinguish between evidence, interpretation, and recommendation?",
    "Can a reader understand the report's central message without reading the full document?"
]

tonality_questions = [
    "Does the summary use a formal academic tone?",
    "Does the summary avoid casual or promotional language?",
    "Does the summary use precise business and AI implementation vocabulary?",
    "Does the summary maintain a neutral and analytical style?",
    "Does the summary avoid exaggerated claims about GenAI capabilities?"
]

safety_questions = [
    "Does the summary avoid unsupported factual claims about AI adoption or business performance?",
    "Does the summary avoid misleading interpretations of the report's findings?",
    "Does the summary avoid presenting speculative claims as established facts?",
    "Does the summary avoid harmful or irresponsible business advice?",
    "Does the summary avoid overstating the certainty or generalizability of the report's conclusions?"
]

def make_criteria(metric_name, questions):
    question_text = "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
    return f"""
Evaluate the summary according to the following {metric_name} questions:

{question_text}

Give a score from 0 to 1 and explain the reason briefly.
"""

def evaluate_summary(summary_to_evaluate):
    test_case = LLMTestCase(
        input=evaluation_source_text,
        actual_output=summary_to_evaluate
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=evaluation_model,
        assessment_questions=summarization_questions,
        include_reason=True
    )

    coherence_metric = GEval(
        name="Coherence",
        criteria=make_criteria("coherence", coherence_questions),
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=evaluation_model
    )

    tonality_metric = GEval(
        name="Tonality",
        criteria=make_criteria("tonality", tonality_questions),
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=evaluation_model
    )

    safety_metric = GEval(
        name="Safety",
        criteria=make_criteria("safety", safety_questions),
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=evaluation_model
    )

    print("Running summarization metric...")
    summarization_metric.measure(test_case)

    print("Running coherence metric...")
    coherence_metric.measure(test_case)

    print("Running tonality metric...")
    tonality_metric.measure(test_case)

    print("Running safety metric...")
    safety_metric.measure(test_case)

    result = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }

    return result

evaluation_result = evaluate_summary(summary_text)

pd.DataFrame([evaluation_result])

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
import json

enhancement_developer_instructions = """
You are an expert editor and evaluator. Your task is to improve an existing article summary
using the original article and the evaluation feedback. Preserve factual accuracy, improve weak areas,
and maintain the requested tone.
"""

enhancement_user_prompt_template = """
Please improve the summary below using the original article and the evaluation feedback.

Required tone:
{tone}

Original article:
{article_text}

Original summary:
{original_summary}

Evaluation feedback:
{evaluation_feedback}

Instructions:
1. Preserve the author's main argument.
2. Improve factual coverage where the evaluation identified weaknesses.
3. Improve coherence and clarity.
4. Maintain the requested tone.
5. Avoid unsupported claims.
6. Keep the enhanced summary concise and under 1000 tokens.
"""

enhancement_user_prompt = enhancement_user_prompt_template.format(
    tone=selected_tone,
    article_text=document_text,
    original_summary=summary_text,
    evaluation_feedback=json.dumps(evaluation_result, indent=2)
)

enhancement_response = client.beta.chat.completions.parse(
    model=model_name,
    messages=[
        {"role": "developer", "content": enhancement_developer_instructions},
        {"role": "user", "content": enhancement_user_prompt}
    ],
    response_format=ArticleSummary,
)

enhanced_summary_obj = enhancement_response.choices[0].message.parsed

enhanced_summary_obj.InputTokens = enhancement_response.usage.prompt_tokens
enhanced_summary_obj.OutputTokens = enhancement_response.usage.completion_tokens

enhanced_summary_text = enhanced_summary_obj.Summary

enhanced_evaluation_result = evaluate_summary(enhanced_summary_text)

comparison = pd.DataFrame([
    {
        "Version": "Original",
        "SummarizationScore": evaluation_result["SummarizationScore"],
        "CoherenceScore": evaluation_result["CoherenceScore"],
        "TonalityScore": evaluation_result["TonalityScore"],
        "SafetyScore": evaluation_result["SafetyScore"],
    },
    {
        "Version": "Enhanced",
        "SummarizationScore": enhanced_evaluation_result["SummarizationScore"],
        "CoherenceScore": enhanced_evaluation_result["CoherenceScore"],
        "TonalityScore": enhanced_evaluation_result["TonalityScore"],
        "SafetyScore": enhanced_evaluation_result["SafetyScore"],
    }
])

print("Enhanced Summary:")
print(enhanced_summary_text)

comparison

The enhanced summary improved because the second generation step used explicit evaluation feedback rather than simply asking the model to summarize the report again. This made the revision more targeted: the model could improve factual coverage, coherence, tone, and safety while preserving the original task requirements.

However, these controls are not sufficient on their own. Since the evaluation is also performed by an LLM, the scores may depend on the evaluator model and may miss subtle factual errors in the report. For a high-stakes business or AI implementation context, I would combine this pipeline with human review, source-grounded citation checks, and more direct factual consistency testing. Still, the evaluation-and-enhancement loop is useful because it creates a more systematic process for improving the quality of LLM-generated summaries.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
